In [1]:
import os
import pickle
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

FILE_PATH = os.path.dirname(os.path.abspath("."))
os.chdir(FILE_PATH)
load_dotenv()
from src.bm25 import BM25Search
from src.semantic import SemanticSearch
from src.hybrid import HybridSearch
from src.tools import web_search
from src.download_data import download_data
os.chdir(f"{FILE_PATH}/notebooks")

download_data()

CATEGORY = "Appliances"
PROCESSED_DATA_DIR = Path("../data/processed")
with open(f"{PROCESSED_DATA_DIR}/{CATEGORY}_product_documents.pkl", "rb") as f:
    documents = pickle.load(f)

with open(f"{PROCESSED_DATA_DIR}/{CATEGORY}_doc_ids.pkl", "rb") as f:
    doc_ids = pickle.load(f)

import duckdb

PROCESSED_DATA_DIR = Path("../data/processed")
product_data_file = "Appliances_products.parquet"

c2 = duckdb.connect()
products = c2.execute(f"SELECT * FROM read_parquet('{PROCESSED_DATA_DIR}/{product_data_file}')").df()

/Users/wnsong/miniforge3/envs/dsci575-project/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Working directory: /Users/wnsong/Documents/MDS/Block6/d575/lab575/DSCI_575_project_hli76_wnsong/src
Review data for Appliances already downloaded
Meta data for Appliances already downloaded
Merged data for Appliances is ready
Products Data for Appliances is ready
Document id for Appliances is ready
Document id for Appliances is ready


In [2]:
print(products.keys())

Index(['parent_asin', 'product_title', 'main_category', 'store', 'price',
       'avg_rating', 'reviews', 'review_titles', 'helpful_votes'],
      dtype='str')


In [3]:
TOP_K = 10



## Create RAG Pipeline using Semantic Retriever

```pseudocode
def pipeline(query):
    retriever = SemanticSearch(Documents)
    retrieved_products = retriever.search(query)
    context = build_docs(retrieved_products)
    prompt = build_prompt(query, context)
    return format_output(llm(prompt))
```

In [4]:
query = "Best container for my food that needs to be cold"

In [5]:
def build_context(results):
    context = ""
    for i, (index, score) in enumerate(results):
        product_asin = doc_ids[index]
        product_context = documents[index]
        # print(f"{i+1}. ({score:.3f}) {product.product_title.values[0]}")
        context += f"""
parent_asin: {product_asin}
{product_context}

"""
    return context

DEFAULT_SYSTEM_PROMPT = """
Instructions:
- You are a helpful Amazon shopping assistant.
- You must answer the question using ONLY the following context (real product reviews with helpful votes and the metadata for the products).
- Always cite the product ASIN when possible.
- If the answer is present, extract and summarize it clearly.
- Do NOT say "I don't know" if the answer exists in the context.
- Only say "I don't know" if the context truly does not contain the answer.
"""

def build_prompt(query, context, system_prompt=DEFAULT_SYSTEM_PROMPT):
    return f"""
{system_prompt}

---------

Context: 
{context}

---------

Question:
{query}

"""

def load_retrievers(documents):
    retrievers = {
        "bm25": BM25Search(documents), 
        "semantic": SemanticSearch(documents)
    }
    retrievers["hybrid"] = HybridSearch(
        bm25=retrievers["bm25"], 
        semantic=retrievers["semantic"], 
        alpha=0.5, 
        top_k_candidates=TOP_K + 100
    )
    return retrievers

llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=os.getenv("GROQ_API_KEY"))

def RAG_pipeline(retriever, documents, query, llm, top_k=TOP_K):
    retriever = load_retrievers(documents)[retriever]
    if isinstance(retriever, HybridSearch):
        raw_results = retriever.search(query, top_k=top_k)
        results = [(idx, score) for idx, score, _details in raw_results]
    else:
        results = retriever.search(query, top_k=top_k)
    context = build_context(results)
    prompt = build_prompt(query, context, system_prompt=DEFAULT_SYSTEM_PROMPT)
    response = llm.invoke(prompt).content
    return response

print(RAG_pipeline("hybrid", documents, query, llm))

load bm25 index
done
load tokenized products
done


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11459.54it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


load faiss index
done
Based on the reviews, a good container for keeping food cold is the IceeNOW Freezer Pak (B08JCZ4WNP) which is a portable cooler bag with non-toxic freezable gel that can keep items cold for hours. 

Another option is the Friomex Dry Ice Packs (B09Q8T47SM) which are reusable and can be used to keep food cold for a long time. They are also easy to use, just soak them in water and put them in the freezer. 

Additionally, the GiTenvy 15 Quarts Portable Car Cooler (B0B3937GVL) is an electric cooler that can be plugged into a car's 12V DC outlet and can keep food and drinks cold for a long time. 

It's worth noting that the best container for you will depend on your specific needs and preferences, so it's a good idea to read the reviews and product descriptions carefully to find the one that best fits your needs.


In [11]:
llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=os.getenv("GROQ_API_KEY"))

def load_retrievers(documents):
    """
    Initialize all retrieval systems.

    Parameters
    ----------
    documents : list
        List of documents used to build BM25, semantic, and hybrid retrievers.

    Returns
    -------
    dict
        Dictionary containing initialized retrievers:
        - "bm25"
        - "semantic"
        - "hybrid"
    """
    retrievers = {
        "bm25": BM25Search(documents), 
        "semantic": SemanticSearch(documents)
    }
    retrievers["hybrid"] = HybridSearch(
        bm25=retrievers["bm25"],
        semantic=retrievers["semantic"],
        alpha=0.5,
        top_k_candidates=TOP_K + 100,
    )
    return retrievers


def build_context(results, documents, doc_ids, max_chars_per_doc=1500):
    """
    Build a formatted context string from retrieval results.

    Parameters
    ----------
    results : list of tuple
        List of (document_index, score) pairs returned by a retriever.
    documents : list of str
        List of document texts corresponding to indices.
    doc_ids : list of str
        List of product identifiers (product_asin) aligned with documents.

    Returns
    -------
    str
        Formatted context string combining product metadata and text.
    """
    context = ""
    for i, (index, score) in enumerate(results):
        product_asin = doc_ids[index]
        product_context = documents[index][:max_chars_per_doc]
        # print(f"{i+1}. ({score:.3f}) {product.product_title.values[0]}")
        context += f"""
parent_asin: {product_asin}
{product_context}

"""
    return context


DEFAULT_SYSTEM_PROMPT = """
Instructions:
- You are a helpful Amazon shopping assistant.
- You have access to historical product reviews and metadata provided in the Context below.
- Always cite the product ASIN when possible.
- If the question requires up-to-date pricing, live availability, or specs missing from the context, use your `web_search` tool to find it.
- Do NOT say "I don't know" if the answer exists in the context or can be found via web search.

Context:
{context}
"""

# build_prompt becomes absolete if we use a tool-calling agent, but we keep it for now since we are not fully implementing the agent in this codebase. 
# The agent implementation is more complex and would require changes to how we structure the prompt and handle tool calls, so we will leave that as a future enhancement.
def build_prompt(query, context, system_prompt=DEFAULT_SYSTEM_PROMPT):
    """
    Construct the final prompt for the LLM.

    Parameters
    ----------
    query : str
        User query.
    context : str
        Retrieved context string from search results.
    system_prompt : str, optional
        System instructions defining model behavior.

    Returns
    -------
    str
        Fully formatted prompt for the language model.
    """
    return f"""
{system_prompt}

---------

Context: 
{context}

---------

Question:
{query}

"""


def RAG_pipeline(retriever, documents, doc_ids, query, llm, top_k=TOP_K):
    """
    Execute a Retrieval-Augmented Generation (RAG) pipeline utilizing an LLM agent 
    with tool-calling capabilities.

    This pipeline first retrieves relevant historical product reviews and metadata using the 
    specified retrieval system. It then constructs a LangChain Agent equipped with a web search 
    tool, allowing the LLM to dynamically fetch up-to-date information if the historical context 
    is insufficient to answer the user's query.

    Parameters
    ----------
    retriever : BM25Search, SemanticSearch, or HybridSearch
        The initialized search system used to retrieve relevant local documents.
    documents : list of str
        The full corpus of product documents/reviews.
    doc_ids : list of str
        The unique identifiers (e.g., parent ASINs) corresponding to the documents.
    query : str
        The question or search input provided by the user.
    llm : BaseChatModel
        The initialized LangChain chat model (e.g., ChatGroq) capable of tool calling.
    top_k : int, optional
        The number of top document candidates to retrieve for context. Defaults to TOP_K.

    Returns
    -------
    str
        The final generated response from the agent, synthesizing both the local historical 
        review context and any live web data retrieved during execution.

    Notes
    -----
    - The function utilizes `create_tool_calling_agent` and `AgentExecutor` to handle 
      multi-step reasoning.
    - The `agent_scratchpad` in the prompt template is required for the LLM to store 
      and read the outputs of its tool calls before formulating the final answer.
    - Currently provisions the `web_search` tool (powered by Tavily) to handle live 
      pricing, availability, and specs queries.
    """
    if isinstance(retriever, HybridSearch):
        raw_results = retriever.search(query, top_k=top_k)
        results = [(idx, score) for idx, score, _details in raw_results]
    else:
        results = retriever.search(query, top_k=top_k)
        
    context = build_context(results, documents, doc_ids)

    tools = [web_search]

    system_prompt = DEFAULT_SYSTEM_PROMPT.replace("{context}", context)

    agent = create_agent(model=llm, tools=tools, system_prompt=system_prompt)

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query},
    ]

    resp = agent.invoke({"messages": messages})

    answer = resp.get("output") or resp.get("result") or str(resp)
    return answer

retrievers = load_retrievers(documents)

retriever_obj = retrievers["hybrid"]

print(RAG_pipeline(retriever_obj, documents, doc_ids, query, llm))

load bm25 index
done
load tokenized products
done


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4619.23it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


load faiss index
done
{'messages': [SystemMessage(content='\nInstructions:\n- You are a helpful Amazon shopping assistant.\n- You have access to historical product reviews and metadata provided in the Context below.\n- Always cite the product ASIN when possible.\n- If the question requires up-to-date pricing, live availability, or specs missing from the context, use your `web_search` tool to find it.\n- Do NOT say "I don\'t know" if the answer exists in the context or can be found via web search.\n\nContext:\n\nparent_asin: B08P4J57J7\nTitle: Plastic Refrigerator Egg Storage Container with Flip-top Lid,Stackable Egg Storage Box with 12 Egg Grooves Kithchen Food Storage Box Plastic Refrigerator Egg Storage Container with Flip-top Lid,Stackable Egg Storage Box with 12 Egg Grooves Kithchen Food Storage Box\nCategory: Tools & Home Improvement\nStore: SCWBOEII\nPrice: nan\nAverage Rating: 4.6\n\nReviews:\n- (0 votes) Is it wrong to buy an egg container just for the aesthetics?: I have a sma